In [ ]:
!pip install open3d datasets plyfile huggingface_hub plotly wandb scikit-image tqdm -q

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 43.3/43.3 kB 2.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 447.7/447.7 MB 1.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 7.2/7.2 MB 126.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 139.8/139.8 kB 16.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.2/2.2 MB 108.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.6/1.6 MB 91.4 MB/s eta 0:00:00


In [ ]:
import os
from google.colab import userdata

os.environ['WANDB_API_KEY'] = userdata.get('WANDB_API_KEY')
os.environ['HF_TOKEN'] = userdata.get('HUGGINGFACE')

from huggingface_hub import hf_hub_download

hf_hub_download(
    repo_id="BGLab/AgriField3D",
    filename="datasets/FielGrwon_ZeaMays_RawPCD_10k.zip",
    repo_type="dataset",
    local_dir="/content/MaizeField3D"
)

datasets/FielGrwon_ZeaMays_RawPCD_10k.zi(…):   0%|          | 0.00/218M [00:00<?, ?B/s]

'/content/MaizeField3D/datasets/FielGrwon_ZeaMays_RawPCD_10k.zip'

In [ ]:
import zipfile

zip_path = "/content/MaizeField3D/datasets/FielGrwon_ZeaMays_RawPCD_10k.zip"
with zipfile.ZipFile(zip_path, 'r') as zip_ref:
    zip_ref.extractall("./MaizeField3D/extracted")

print("Unzipped!")

Unzipped!


In [ ]:
import os

data_path = "/content/MaizeField3D/extracted"
ply_files = []

for root, dirs, files in os.walk(data_path):
    for file in files:
        if file.endswith(".ply"):
            ply_files.append(os.path.join(root, file))

print(f"Total PLY files: {len(ply_files)}")

Total PLY files: 1045


#1. Leaf Surface Area


More leaf surface area → more light absorption → more photosynthesis.

In [ ]:
import open3d as o3d

def pointcloud_to_mesh(ply_file):

    pcd = o3d.io.read_point_cloud(ply_file)

    pcd.estimate_normals()

    mesh,_ = o3d.geometry.TriangleMesh.create_from_point_cloud_poisson(
        pcd,
        depth=8
    )

    return mesh

#Compute Surface Area

In [ ]:
def compute_leaf_area(mesh):

    return mesh.get_surface_area()

#2. Lead Angle Distribution

Leaves capures sunlight best when not vertical

In [ ]:
import numpy as np

def leaf_angle_score(ply_file):

    pcd = o3d.io.read_point_cloud(ply_file)

    pcd.estimate_normals()

    normals = np.asarray(pcd.normals)

    vertical = np.array([0,0,1])

    angles = np.arccos(
        np.clip(normals @ vertical, -1,1)
    )

    angles = np.degrees(angles)

    score = np.mean(
        (angles > 30) & (angles < 60)
    )

    return score

#3. Light Interception Simulation

Simulating sun rays hitting the plant
More intercepted rays --> higher photosynthesis

In [ ]:
import numpy as np

def light_interception_score(ply_file):

    pcd = o3d.io.read_point_cloud(ply_file)

    points = np.asarray(pcd.points)

    sun_dir = np.array([0,0,-1])

    score = np.sum(points[:,2] > np.percentile(points[:,2],70))

    score = score / len(points)

    return score

This approximates top canopy exposure.

#4. Creating score label for all plants

`score =
0.5 * leaf_area_score +
0.3 * leaf_angle_score +
0.2 * light_interception_score`

In [ ]:
labels = []
ply_files=ply_files[:500]
for file in ply_files:

    area = compute_leaf_area(pointcloud_to_mesh(file))
    angle = leaf_angle_score(file)
    light = light_interception_score(file)

    score = 0.5*area + 0.3*angle + 0.2*light

    labels.append(score)

Normalize

In [ ]:
labels = np.array(labels)
labels = (labels - labels.min())/(labels.max()-labels.min())

# DGCNN implementation

In [ ]:
!pip install open3d torch torchvision torchaudio numpy scikit-learn tqdm -q

In [ ]:
import open3d as o3d
import numpy as np
import glob
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader

In [ ]:
def load_pointcloud(file, n_points=4096):

    pcd = o3d.io.read_point_cloud(file)

    pcd.estimate_normals(
        search_param=o3d.geometry.KDTreeSearchParamHybrid(
            radius=0.05,
            max_nn=30
        )
    )

    pts = np.asarray(pcd.points)
    normals = np.asarray(pcd.normals)

    data = np.concatenate([pts, normals], axis=1)

    idx = np.random.choice(len(data), n_points, replace=len(data) < n_points)

    data = data[idx]

    # normalize coordinates
    data[:,:3] -= data[:,:3].mean(0)
    data[:,:3] /= np.max(np.linalg.norm(data[:,:3], axis=1))

    return data.astype(np.float32)

In [ ]:
class PlantDataset(Dataset):

    def __init__(self, files, labels):

        self.files = files
        self.labels = labels

    def __len__(self):

        return len(self.files)

    def __getitem__(self, idx):

        pts = load_pointcloud(self.files[idx])

        pts = torch.tensor(pts).transpose(0,1)   # [6,4096]

        label = torch.tensor(self.labels[idx]).float()

        return pts, label

In [ ]:
dataset = PlantDataset(ply_files, labels)

loader = DataLoader(
    dataset,
    batch_size=8,
    shuffle=True,
    num_workers=2
)

##KNN GRAPH FUNCTION

In [ ]:
def knn(x, k):

    inner = -2 * torch.matmul(x.transpose(2,1), x)
    xx = torch.sum(x**2, dim=1, keepdim=True)
    pairwise = -xx - inner - xx.transpose(2,1)

    idx = pairwise.topk(k=k, dim=-1)[1]

    return idx

In [ ]:
def get_graph_feature(x, idx):

    B, C, N = x.shape
    k = idx.shape[-1]

    idx_base = torch.arange(0, B, device=x.device).view(-1,1,1) * N
    idx = idx + idx_base
    idx = idx.view(-1)

    x = x.transpose(2,1).contiguous()

    feature = x.view(B*N, C)[idx,:]
    feature = feature.view(B, N, k, C)

    x = x.view(B, N, 1, C).repeat(1,1,k,1)

    edge_feature = torch.cat((x, feature-x), dim=3)

    return edge_feature.permute(0,3,1,2).contiguous()


class EdgeConv(nn.Module):

    def __init__(self, in_channels, out_channels):

        super().__init__()

        self.mlp = nn.Sequential(
            nn.Conv2d(in_channels*2, out_channels, 1),
            nn.BatchNorm2d(out_channels),
            nn.ReLU()
        )

    def forward(self, x, idx):

        edge = get_graph_feature(x, idx)

        out = self.mlp(edge)

        out = out.max(dim=-1)[0]

        return out

##MODEL

In [ ]:
class DGCNN_Photosynthesis(nn.Module):

    def __init__(self, k=20):

        super().__init__()

        self.k = k

        self.conv1 = EdgeConv(6,64)
        self.conv2 = EdgeConv(64,64)
        self.conv3 = EdgeConv(64,128)
        self.conv4 = EdgeConv(128,256)

        self.fc = nn.Sequential(
            nn.Linear(256,128),
            nn.ReLU(),
            nn.Linear(128,1)
        )

    def forward(self,x):

        idx = knn(x,self.k)

        x = self.conv1(x,idx)
        x = self.conv2(x,idx)
        x = self.conv3(x,idx)
        x = self.conv4(x,idx)

        x = torch.max(x,dim=2)[0]

        return self.fc(x)

In [ ]:
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

model = DGCNN_Photosynthesis().to(device)
optimizer = torch.optim.Adam(model.parameters(), lr=0.001)

criterion = nn.MSELoss()

In [ ]:
epochs = 100

for epoch in range(epochs):

    model.train()

    total_loss = 0

    for pts,label in loader:

        pts = pts.to(device)
        label = label.to(device)

        pred = model(pts).squeeze()

        loss = criterion(pred,label)

        optimizer.zero_grad()
        loss.backward()
        optimizer.step()

        total_loss += loss.item()

    print("Epoch:",epoch,"Loss:",total_loss/len(loader))

Epoch: 0 Loss: 0.5333614253719884
Epoch: 1 Loss: 0.03912546407313101
Epoch: 2 Loss: 0.0410024318548422
Epoch: 3 Loss: 0.021175074176715006
Epoch: 4 Loss: 0.02418169355581677
Epoch: 5 Loss: 0.023009835601976468
Epoch: 6 Loss: 0.019342700074914666
Epoch: 7 Loss: 0.015458898397073859
Epoch: 8 Loss: 0.0186038100648494
Epoch: 9 Loss: 0.016196233656493916
Epoch: 10 Loss: 0.01608033242472817
Epoch: 11 Loss: 0.01474200318833547
Epoch: 12 Loss: 0.011303440974433981
Epoch: 13 Loss: 0.01205517550457328
Epoch: 14 Loss: 0.015073872165798786
Epoch: 15 Loss: 0.012734056764563162
Epoch: 16 Loss: 0.010862048087079846
Epoch: 17 Loss: 0.00901012347515909
Epoch: 18 Loss: 0.011898173148640327
Epoch: 19 Loss: 0.010490051134385996
Epoch: 20 Loss: 0.01145337508355696
Epoch: 21 Loss: 0.010985419520043902
Epoch: 22 Loss: 0.011355241076902502
Epoch: 23 Loss: 0.008732431146892764
Epoch: 24 Loss: 0.009099125669943907
Epoch: 25 Loss: 0.009883122357894621
Epoch: 26 Loss: 0.0070885310969537214
Epoch: 27 Loss: 0.00687

In [ ]:
torch.save({
    "epoch": epoch,
    "model_state_dict": model.state_dict(),
    "optimizer_state_dict": optimizer.state_dict(),
    "loss": loss,
}, "checkpoint.pth")

In [ ]:
torch.save(model.state_dict(), "model.pth")

In [ ]:
def predict_photosynthesis(file):

    model.eval()

    pts = load_pointcloud(file)

    pts = torch.tensor(pts).unsqueeze(0).transpose(1,2).to(device)

    with torch.no_grad():

        score = model(pts)

    return float(score.item())

In [ ]:
import open3d as o3d
import numpy as np
import plotly.graph_objects as go

def visualize_plant(ply_file):
    pcd = o3d.io.read_point_cloud(ply_file)
    points = np.asarray(pcd.points)

    fig = go.Figure(
        data=[
            go.Scatter3d(
                x=points[:,0],
                y=points[:,1],
                z=points[:,2],
                mode="markers",
                marker=dict(size=2)
            )
        ]
    )

    fig.update_layout(
        scene=dict(
            xaxis_title="X",
            yaxis_title="Y",
            zaxis_title="Z"
        ),
        height=700
    )

    fig.show()

visualize_plant(ply_files[400])

In [ ]:
predict_photosynthesis(ply_files[400])

0.5990542769432068

#ReAct Agent

In [ ]:
import os
import re
import openai
import torch
import torch.nn as nn
import open3d as o3d
import numpy as np

# =========================
# 1. API KEY
# =========================
from google.colab import userdata
os.environ["OPENAI_API_KEY"] = userdata.get("OPENAI_API_KEY")

client = openai.OpenAI()
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

In [ ]:
def photosynthesis_tool(ply_file):
    score = predict_photosynthesis(ply_file)

    if score > 0.7:
        quality = "High photosynthesis potential"
    elif score > 0.4:
        quality = "Moderate photosynthesis potential"
    else:
        quality = "Low photosynthesis potential"

    return f"Photosynthesis score: {score:.3f}. {quality}"

In [ ]:
SYSTEM_PROMPT = """
You are a STRICT ReAct agent.

Rules:
1. You MUST use the tool to answer questions about plants.
2. Do NOT guess or hallucinate values.
3. You MUST follow this EXACT format:

Question: ...
Thought: ...
Action: predict_photosynthesis[path_to_ply]
Observation: ...
Thought: ...
Final Answer: ...

4. If you already have the Observation, directly give Final Answer.
7. You MUST NOT include 'Observation:', 'Thought:', or 'Final Answer:' placeholders in your response if you are calling a tool.
5. Maximum 1 tool call is enough.
6. If no valid action → STOP.

Available tools:
predict_photosynthesis: takes a .ply file path
"""

In [ ]:
def parse_action(text):
    match = re.search(r"Action:\s*predict_photosynthesis\[(.*?)\]", text)
    if match:
        return match.group(1).strip()
    return None

# =========================
# 7. REACT LOOP
# =========================
def react_agent(question, max_steps=3):
    messages = [
        {"role": "system", "content": SYSTEM_PROMPT},
        {"role": "user", "content": f"Question: {question}"}
    ]

    for step in range(max_steps):
        response = client.chat.completions.create(
            model="gpt-4o-mini",
            messages=messages,
            temperature=0,
            stop=["Observation:"]
        )

        reply = response.choices[0].message.content
        print(f"\n--- Step {step+1} ---\n{reply}")

        if "Final Answer:" in reply:
            return reply

        tool_input = parse_action(reply)

        if tool_input is None:
            return "Stopped: No valid action (prevents hallucination)"

        try:
            observation = photosynthesis_tool(tool_input)
        except Exception as e:
            return f"Tool Error: {str(e)}"

        messages.append({"role": "assistant", "content": reply})
        messages.append({
            "role": "user",
            "content": (
                f"Observation: {observation}\n\n"
                "You MUST now produce:\n"
                "Final Answer: <based ONLY on the observation above>\n\n"
                "Do NOT write Thought or Action.\n"
                "Do NOT leave placeholders.\n"
                "Do NOT say '...'."
            )
        })

    return "Stopped: Max steps reached (safe cutoff)"

In [ ]:
# =========================
# 8. RUN
# =========================
react_agent("Check if this maize plant is good for photosynthesis: /content/MaizeField3D/extracted/FielGrwon_ZeaMays_RawPCD_10k/0005.ply")


--- Step 1 ---
Thought: I need to analyze the provided .ply file to determine the photosynthesis potential of the maize plant. 
Action: predict_photosynthesis[/content/MaizeField3D/extracted/FielGrwon_ZeaMays_RawPCD_10k/0005.ply]

--- Step 2 ---
Final Answer: The maize plant has a low photosynthesis potential with a score of 0.201.


'Final Answer: The maize plant has a low photosynthesis potential with a score of 0.201.'